In [ ]:
import os
import sys

%load_ext autoreload
%autoreload 2
import logging

import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt


# Configure logging
logging.basicConfig(level=logging.INFO)

# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, "..", "src"))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.append(parent_dir)
import pickle
from server_config import (
    datapath,
    preprocessed_path_freezed,
    redcap_path,
    preprocessed_path,
)
from functions.preprocessing.aggregation import compute_sleep_sessions


In [ ]:
# backup_path = preprocessed_path_freezed + "/backup_data_passive_actual.feather"
backup_path = (
    "/sc-projects/sc-proj-cc15-preact/SP6/raw/backup_passive_recent.feather"  # new file
)
df_backup = pd.read_feather(backup_path)
print(df_backup.shape)
assert "local_start_time" in df_backup.columns
assert "local_end_time" in df_backup.columns

df_backup.head()

# Sleep

_inspired by @RADAR_

| **No.** | **Column Name** | **Description** | 
| --| -- | -- |
| 1 | `customer` | Unique identifier for the customer/user | 
| 2 | `sleep_session_id` | Unique identifier for the specific sleep session | 
| 3 | `startTimestamp` | UTC timestamp marking the start of sleep session | 
| 4 | `endTimestamp` | UTC timestamp marking the end of sleep session | 
| 5 | `local_start_time` | Local time string/timestamp for the start of sleep session| 
| 6 | `local_end_time` | Local time string/timestamp for the end of sleep session | 
| 7 | `sleep_session_duration` | Total duration of the sleep session (end - start)| 
| 8 | `SleepAwake_duration` | Total duration spent in the awake stage | 
| 9 | `SleepDeep_duration` | Total duration spent in the deep sleep stage | 
| 10 | ~~`SleepInBed_duration`~~ | ~~Total duration spent in bed~~ (DO NOT USE - it isn't recorded for initial participants) | 
| 11 | `SleepLight_duration` | Total duration spent in the light sleep stage | 
| 12 | `total_sleep_time` | Total sleep time, i.e., sum of all "non-awake" stages, in seconds | 
| 13 | `awakenings` | Number of episodes in which the participant is awake for more than 5 minutes | 
| 14 | `long_awakenings` | Number of long awakening episodes (longer than 30 minutes) | 
| 15 | `sleep_onset` | Timestamp for the onset of sleep | 
| 16 | `sleep_offset` | Timestamp for the offset of sleep | 
| 17 | `sleep_onset_hour` | Exact onset time (in hours local time as decimal number, e.g., 22.5 = 10:30 p.m.) of sleep record | 
| 18 | `sleep_offset_hour` | Exact offset time (in hours local time as decimal number, e.g., 7.25 = 7:15 a.m.) of sleep record | 
| 19 | `time_in_bed` | Total time in bed, i.e., sum of all detected stages (including awake stages), in seconds | 
| 20 | `time_out_of_bed` | Total time spent out of bed during the session | 
| 21 | `sleep_efficiency` | Percentage of total sleep time to time in bed | 
| 22 | `hypersomnia` | flag if total sleep time > 10 hours | 
| 23 | `insomnia` | flag if total sleep time < 6 hours and at least one awakening of more than 30 minutes | 
| 24 | `awake_pct` | Proportion of total time in bed spent awake | 
| 25 | `light_sleep_pct` | Proportion of total time in bed spent in light sleep stage | 
| 26 | `deep_sleep_pct` | Proportion of total time in bed spent in deep sleep stage | 
| 27 | `day` | Day of the sleep session (day of the wake-up time)| 
| 28 | `num_sessions_in_day` | Count of distinct sleep sessions recorded for this customer on this day |

In [ ]:
df_sleep_sessions = compute_sleep_sessions(df_backup)
df_sleep_sessions.head()

more than 90% of the customer-days got only one session, but some have more, what to do later?
- only analyze one, longest sleep session
- aggregate sessions further by summing over relevant columns

In [ ]:
# new df with just longest sleep session per customer per day
df_sleep_sessions_longest = df_sleep_sessions.loc[
    df_sleep_sessions.groupby(["customer", "day"])["sleep_session_duration"].idxmax()
].reset_index(drop=True)
# df_sleep_sessions_longest.groupby(["customer", "day"]).size().value_counts()
df_sleep_sessions_longest.head()

In [ ]:
df_sleep_sessions.groupby(["customer", "day"]).agg(
    {
        "sleep_session_duration": "sum",
        "total_sleep_time": "sum",
        "time_in_bed": "sum",
        "time_out_of_bed": "sum",
        "SleepAwake_duration": "sum",
        "SleepLight_duration": "sum",
        "SleepDeep_duration": "sum",
        "awakenings": "sum",
        "long_awakenings": "sum",
        "num_sessions_in_day": "first",  # same for all sessions in that day
    }
)